Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd

query = """

WITH base AS (
    SELECT
        m.sub_ticket_id,
        m.ticket_id,
        m.canonical_ticket_id,
        m.customer_id,
        m.ticket_group,
        m.ticket_state,
        m.source_channel,
        m.overall_last_agent_type_for_sub_ticket,
        m.last_obcs_l1_agent_id,
        m.last_obcs_l1_agent_name,
        m.overall_last_agent_id_for_sub_ticket,
        m.overall_last_agent_name_for_sub_ticket,
        m.ticket_created_dt,
        m.phone_number,
        m.ticket_created_time,
        m.resolved_time,
        m.last_agent_id_for_reporting,
        m.last_agent_name_for_reporting,
        o.origination_reason,
        o.origination_sub_reason

    FROM (
        SELECT
            sub_ticket_id,
            ticket_id,
            canonical_ticket_id,
            customer_id,
            ticket_group,
            ticket_state,
            source_channel,
            overall_last_agent_type_for_sub_ticket,
            last_obcs_l1_agent_id,
            last_obcs_l1_agent_name,
            overall_last_agent_id_for_sub_ticket,
            overall_last_agent_name_for_sub_ticket,
            ticket_created_dt,
            phone_number,
            ticket_created_time,
            resolved_time,

            CASE
                WHEN overall_last_agent_type_for_sub_ticket = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
                    THEN coalesce(overall_last_agent_id_for_sub_ticket, last_obcs_l1_agent_id)
                ELSE last_obcs_l1_agent_id
            END AS last_agent_id_for_reporting,

            CASE
                WHEN overall_last_agent_type_for_sub_ticket = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
                    THEN coalesce(overall_last_agent_name_for_sub_ticket, last_obcs_l1_agent_name)
                ELSE last_obcs_l1_agent_name
            END AS last_agent_name_for_reporting

        FROM corp_analytics.cx_sub_ticket_data_model_tbl

        WHERE ticket_created_dt >= date_format(current_date - interval '30' day, '%Y%m%d')
          AND overall_last_agent_type_for_sub_ticket = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
    ) m

    LEFT JOIN (
        SELECT
            tk.canonical_ticket_id,
            arbitrary(tk.origination_reason) AS origination_reason,
            arbitrary(tk.origination_sub_reason) AS origination_sub_reason

        FROM crm_timekeeper.ticket_event_audit t

        INNER JOIN crm_tickets.tickets tk
            ON t.ticket_id = tk.id

        WHERE tk.agent_type = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
          AND t.dispositions IS NOT NULL

        GROUP BY tk.canonical_ticket_id
    ) o
        ON CAST(m.canonical_ticket_id AS varchar) = CAST(o.canonical_ticket_id AS varchar)
),

crm_l1_excluded AS (
    SELECT DISTINCT
        CAST(tk.id AS varchar) AS crm_ticket_id,
        CAST(tk.canonical_ticket_id AS varchar) AS canonical_ticket_id
    FROM crm_tickets.tickets tk
    LEFT JOIN crm_timekeeper.ticket_event_audit t
        ON t.ticket_id = tk.id
    WHERE tk.agent_type = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
      AND (
            lower(trim(CAST(tk.origination_reason AS varchar))) IN (
                'not connected',
                'non claim related'
            )
         OR lower(trim(CAST(tk.origination_sub_reason AS varchar))) IN (
                'short call',
                'others',
                'other',
                'non insurance related',
                'loan related',
                'upi related',
                'test call',
                'rnr / busy',
                'rnr/ busy',
                'rnr/switched off/ busy/ not reachalbe',
                'rnr/switched off/ busy/ not reachable',
                'rnr/switched off',
                'upi /trezo / bbps',
                'cash loan / pl',
                'home loan',
                'not connected /rnr/swithch off /busy'
            )
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%not connected%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%non claim related%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%short call%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%others%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%non insurance related%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%loan related%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%upi related%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%test call%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%rnr/ busy%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%rnr / busy%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%rnr/switched off%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%upi /trezo / bbps%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%cash loan / pl%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%home loan%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%not connected /rnr/swithch off /busy%'
      )
),

fd_report AS (
    SELECT
        ticket_id,
        arbitrary(claims_lob) AS claims_lob,
        arbitrary(claims_type) AS claims_type,
        arbitrary(call_status) AS call_status,
        arbitrary(call_email_disposition) AS call_email_disposition
    FROM omnitrix.freshdesk_ticket_gi_report
    GROUP BY ticket_id
),

-- Query 2 CTEs (GI outbound connected calls)
gi_outbound_tickets AS (
    SELECT
        t.id AS ticket_id,
        t.uuid AS ticket_uuid,
        json_extract_scalar(t.meta_data, '$.call_id') AS call_id,
        json_extract_scalar(t.meta_data, '$.call_flow') AS call_flow,
        date(
            from_iso8601_timestamp(CAST(t.created_at AS varchar)) AT TIME ZONE 'Asia/Kolkata'
        ) AS call_date_ist,
        from_iso8601_timestamp(CAST(t.created_at AS varchar)) AT TIME ZONE 'Asia/Kolkata' AS call_timestamp_ist,
        t.created_at,
        t.ticket_group,
        t.agent_type,
        t.source_channel,
        t.ticket_state,
        CASE
            WHEN json_extract_scalar(t.meta_data, '$.call_flow') = 'INBOUND' THEN 'INBOUND'
            ELSE 'OUTBOUND'
        END AS call_direction
    FROM crm_tickets.tickets t
    WHERE t.ticket_group = 'GENERAL_INSURANCE'
      AND t.agent_type = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
      AND t.source_channel = 'CALL'
),

gi_outbound_agent_lookup AS (
    SELECT
        m.ticket_id,
        max(m.overall_last_agent_id_for_ticket) AS agent_id,
        max(m.overall_last_agent_type_for_ticket) AS overall_last_agent_type_for_ticket,
        max(m.canonical_ticket_id) AS canonical_ticket_id
    FROM corp_analytics.cx_master_data_model_tbl m
    WHERE m.partition_key >= date_format(current_date - interval '120' day, '%Y%m%d')
    GROUP BY m.ticket_id
),

gi_outbound_calls AS (
    SELECT
        t.ticket_id,
        t.ticket_uuid,
        al.canonical_ticket_id,
        t.ticket_group AS ticket_group_name,
        t.call_id,
        t.call_flow,
        t.call_date_ist,
        t.call_timestamp_ist,
        t.call_direction,
        t.ticket_state,
        al.overall_last_agent_type_for_ticket,
        u.name AS agent_name,
        u.email_id AS agent_email,
        cl.connected_at_timestamp,
        date(
            at_timezone(
                coalesce(
                    try(from_iso8601_timestamp(CAST(cl.connected_at_timestamp AS varchar))),
                    cast(cl.connected_at_timestamp AS timestamp)
                ),
                'Asia/Kolkata'
            )
        ) AS resolved_date,
        at_timezone(
            coalesce(
                try(from_iso8601_timestamp(CAST(cl.connected_at_timestamp AS varchar))),
                cast(cl.connected_at_timestamp AS timestamp)
            ),
            'Asia/Kolkata'
        ) AS resolved_timestamp_ist,
        CASE
            WHEN t.call_direction = 'OUTBOUND'
                 AND cl.connected_at_timestamp IS NOT NULL THEN 'CONNECTED'
            WHEN t.call_direction = 'OUTBOUND' THEN 'NOT CONNECTED'
            ELSE NULL
        END AS outbound_connection_status
    FROM gi_outbound_tickets t
    LEFT JOIN gi_outbound_agent_lookup al
        ON al.ticket_id = t.ticket_id
    LEFT JOIN mjolnir_service.users u
        ON u.account_id = al.agent_id
       AND u.__deleted = 'false'
    INNER JOIN graham_bell_service.call_logs cl
        ON cl.call_id = t.call_id
    WHERE t.call_direction = 'OUTBOUND'
      AND cl.connected_at_timestamp IS NOT NULL
      AND date(
            at_timezone(
                coalesce(
                    try(from_iso8601_timestamp(CAST(cl.connected_at_timestamp AS varchar))),
                    cast(cl.connected_at_timestamp AS timestamp)
                ),
                'Asia/Kolkata'
            )
        ) >= current_date - interval '35' day
),

combined AS (

    -- Branch 1: Existing general Freshdesk groups
    SELECT
        CASE lower(trim(CAST(hit.agent_name AS varchar)))
            WHEN 'durga.sharma@navi.com' THEN 'Durga Prasad Sharma'
            WHEN 'afzal.sayed@navi.com' THEN 'Afzal Sayed'
            ELSE hit.agent_name
        END AS agent_name,

        CASE
            WHEN hit.group_name = 'Claims Support Emails' THEN 'Email'
            ELSE 'Call'
        END AS Channel,

        hit.status,
        hit.policy_number,
        hit.ticket_id,
        hit.claim_id,

        regexp_extract(hit.subject, 'Customer Number: ([+0-9]+)', 1) AS customer_number,

        cast(split_part(replace(hit.created_date, 'T', ' '), '.', 1) AS timestamp)
            AS call_initiated_date_and_time,

        CASE
            WHEN hit.group_name = 'Claims Support Emails'
                 AND hit.status = 'Pending'
            THEN cast(split_part(replace(hit.last_updated_date, 'T', ' '), '.', 1) AS timestamp)

            WHEN hit.group_name = 'Claims Support Emails'
                 AND hit.status = 'Resolved'
            THEN coalesce(
                cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp),
                cast(split_part(replace(hit.closed_date, 'T', ' '), '.', 1) AS timestamp)
            )

            ELSE cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp)
        END AS resolved_date_and_time,

        date(
            CASE
                WHEN hit.group_name = 'Claims Support Emails'
                     AND hit.status = 'Pending'
                THEN cast(split_part(replace(hit.last_updated_date, 'T', ' '), '.', 1) AS timestamp)

                WHEN hit.group_name = 'Claims Support Emails'
                     AND hit.status = 'Resolved'
                THEN coalesce(
                    cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp),
                    cast(split_part(replace(hit.closed_date, 'T', ' '), '.', 1) AS timestamp)
                )

                ELSE cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp)
            END
        ) AS resolved_date,

        hit.group_name,

        fd.claims_lob AS "Claims-LOB",

        fd.call_email_disposition,

        CAST(NULL AS varchar) AS origination_sub_reason

    FROM omnitrix.hi_tickets_v1 hit

    LEFT JOIN fd_report fd
        ON fd.ticket_id = hit.ticket_id

    WHERE date(
            CASE
                WHEN hit.group_name = 'Claims Support Emails'
                     AND hit.status = 'Pending'
                THEN cast(split_part(replace(hit.last_updated_date, 'T', ' '), '.', 1) AS timestamp)

                ELSE coalesce(
                    cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp),
                    cast(split_part(replace(hit.closed_date, 'T', ' '), '.', 1) AS timestamp)
                )
            END
        ) >= current_date - interval '35' day

      AND (
            (
                hit.status IN ('Resolved', 'Closed')
                AND hit.group_name IN (
                    'Senior Callback',
                    'Claims Support',
                    'Rejection Calling',
                    'Claims De-escalation'
                )
            )

            OR (
                hit.status = 'Resolved'
                AND hit.group_name = 'Claims Support Emails'
            )

            OR (
                hit.group_name = 'Claims Support Emails'
                AND hit.status = 'Pending'
                AND coalesce(
                    nullif(trim(CAST(hit.resolved_date AS varchar)), ''),
                    nullif(trim(CAST(hit.closed_date AS varchar)), '')
                ) IS NULL
                AND cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp) IS NULL
                AND cast(split_part(replace(hit.closed_date, 'T', ' '), '.', 1) AS timestamp) IS NULL
                AND cast(split_part(replace(hit.last_updated_date, 'T', ' '), '.', 1) AS timestamp)
                    >= current_date - interval '30' day
                AND NOT EXISTS (
                    SELECT 1
                    FROM omnitrix.freshdesk_ticket_gi_report fd_chk
                    WHERE fd_chk.ticket_id = hit.ticket_id
                      AND (
                          lower(trim(coalesce(CAST(fd_chk.call_email_disposition AS varchar), '')))
                              LIKE '%rejection justification%'
                          OR lower(trim(coalesce(CAST(fd_chk.call_email_disposition AS varchar), '')))
                              LIKE '%rejection rationale%'
                      )
                )
            )
        )

      AND (
            (
                hit.status = 'Resolved'
                AND hit.group_name = 'Claims Support Emails'
            )

            OR (
                (
                    hit.group_name NOT IN ('Rejection Calling', 'Senior Callback')
                    AND fd.claims_type NOT IN ('Not Connected')
                )

                OR (
                    hit.group_name IN ('Rejection Calling', 'Senior Callback')
                    AND fd.call_status = 'Connected'
                )
            )
        )

      AND (
            (
                hit.group_name = 'Claims Support Emails'
                AND hit.status = 'Pending'
            )

            OR coalesce(trim(CAST(hit.agent_name AS varchar)), '') <> 'No Agent'
        )

      AND (
            (
                hit.group_name = 'Claims Support Emails'
                AND hit.status = 'Resolved'
            )

            OR hit.policy_number NOT LIKE 'M%'
        )

      AND NOT EXISTS (
            SELECT 1
            FROM crm_l1_excluded e
            WHERE CAST(hit.ticket_id AS varchar) = e.crm_ticket_id
               OR CAST(hit.ticket_id AS varchar) = e.canonical_ticket_id
        )

    UNION ALL

    -- Branch 2: Motor Claims
    SELECT
        CASE lower(trim(CAST(hit.agent_name AS varchar)))
            WHEN 'durga.sharma@navi.com' THEN 'Durga Prasad Sharma'
            WHEN 'afzal.sayed@navi.com' THEN 'Afzal Sayed'
            ELSE hit.agent_name
        END AS agent_name,

        CASE
            WHEN hit.group_name = 'Claims Support Emails' THEN 'Email'
            ELSE 'Call'
        END AS Channel,

        hit.status,
        hit.policy_number,
        hit.ticket_id,
        hit.claim_id,

        regexp_extract(hit.subject, 'Customer Number: ([+0-9]+)', 1) AS customer_number,

        cast(split_part(replace(hit.created_date, 'T', ' '), '.', 1) AS timestamp)
            AS call_initiated_date_and_time,

        CASE
            WHEN hit.group_name = 'Claims Support Emails'
                 AND hit.status = 'Pending'
            THEN cast(split_part(replace(hit.last_updated_date, 'T', ' '), '.', 1) AS timestamp)

            WHEN hit.group_name = 'Claims Support Emails'
                 AND hit.status = 'Resolved'
            THEN coalesce(
                cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp),
                cast(split_part(replace(hit.closed_date, 'T', ' '), '.', 1) AS timestamp)
            )

            ELSE cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp)
        END AS resolved_date_and_time,

        date(
            CASE
                WHEN hit.group_name = 'Claims Support Emails'
                     AND hit.status = 'Pending'
                THEN cast(split_part(replace(hit.last_updated_date, 'T', ' '), '.', 1) AS timestamp)

                WHEN hit.group_name = 'Claims Support Emails'
                     AND hit.status = 'Resolved'
                THEN coalesce(
                    cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp),
                    cast(split_part(replace(hit.closed_date, 'T', ' '), '.', 1) AS timestamp)
                )

                ELSE cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp)
            END
        ) AS resolved_date,

        'Motor Claims' AS group_name,

        fd.claims_lob AS "Claims-LOB",

        fd.call_email_disposition,

        CAST(NULL AS varchar) AS origination_sub_reason

    FROM omnitrix.hi_tickets_v1 hit

    LEFT JOIN fd_report fd
        ON fd.ticket_id = hit.ticket_id

    WHERE (
            (
                hit.group_name IN ('Claims Support', 'Claims Support Emails')
                AND fd.claims_lob = 'Motor Claims'
            )

            OR hit.group_name IN (
                'Motor Claims - Emails',
                'Motor Claims - Outbound Call'
            )
        )

      AND NOT (
            hit.group_name = 'Claims Support Emails'
            AND hit.status = 'Resolved'
        )

      AND date(
            CASE
                WHEN hit.group_name = 'Claims Support Emails'
                     AND hit.status = 'Pending'
                THEN cast(split_part(replace(hit.last_updated_date, 'T', ' '), '.', 1) AS timestamp)

                ELSE coalesce(
                    cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp),
                    cast(split_part(replace(hit.closed_date, 'T', ' '), '.', 1) AS timestamp)
                )
            END
        ) >= current_date - interval '35' day

      AND (
            (
                hit.status IN ('Resolved', 'Closed')
                AND hit.group_name IN (
                    'Claims Support',
                    'Motor Claims - Outbound Call',
                    'Motor Claims - Emails'
                )
            )

            OR (
                hit.group_name = 'Claims Support Emails'
                AND hit.status = 'Pending'
                AND coalesce(
                    nullif(trim(CAST(hit.resolved_date AS varchar)), ''),
                    nullif(trim(CAST(hit.closed_date AS varchar)), '')
                ) IS NULL
                AND cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp) IS NULL
                AND cast(split_part(replace(hit.closed_date, 'T', ' '), '.', 1) AS timestamp) IS NULL
                AND cast(split_part(replace(hit.last_updated_date, 'T', ' '), '.', 1) AS timestamp)
                    >= current_date - interval '30' day
                AND NOT EXISTS (
                    SELECT 1
                    FROM omnitrix.freshdesk_ticket_gi_report fd_chk
                    WHERE fd_chk.ticket_id = hit.ticket_id
                      AND (
                          lower(trim(coalesce(CAST(fd_chk.call_email_disposition AS varchar), '')))
                              LIKE '%rejection justification%'
                          OR lower(trim(coalesce(CAST(fd_chk.call_email_disposition AS varchar), '')))
                              LIKE '%rejection rationale%'
                      )
                )
            )
        )

      AND NOT EXISTS (
            SELECT 1
            FROM crm_l1_excluded e
            WHERE CAST(hit.ticket_id AS varchar) = e.crm_ticket_id
               OR CAST(hit.ticket_id AS varchar) = e.canonical_ticket_id
        )

    UNION ALL

    -- Branch 3: BIMA Bharosa Portal & Grievance Team (original fd_bbp join)
    SELECT
        CASE lower(trim(CAST(hit.agent_name AS varchar)))
            WHEN 'durga.sharma@navi.com' THEN 'Durga Prasad Sharma'
            WHEN 'afzal.sayed@navi.com' THEN 'Afzal Sayed'
            ELSE hit.agent_name
        END AS agent_name,

        'Call' AS Channel,

        hit.status,
        hit.policy_number,
        hit.ticket_id,
        hit.claim_id,

        regexp_extract(hit.subject, 'Customer Number: ([+0-9]+)', 1) AS customer_number,

        cast(split_part(replace(hit.created_date, 'T', ' '), '.', 1) AS timestamp)
            AS call_initiated_date_and_time,

        cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp)
            AS resolved_date_and_time,

        date(
            cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp)
        ) AS resolved_date,

        hit.group_name,

        fd_bbp.claims_lob AS "Claims-LOB",

        fd_bbp.call_email_disposition,

        CAST(NULL AS varchar) AS origination_sub_reason

    FROM omnitrix.hi_tickets_v1 hit

    LEFT JOIN omnitrix.freshdesk_ticket_gi_report fd_bbp
        ON fd_bbp.ticket_id = hit.ticket_id

    WHERE hit.group_name IN ('BIMA Bharosa Portal', 'Grievance Team')

      AND hit.status IN ('Resolved', 'Closed')

      AND date(
            coalesce(
                cast(split_part(replace(hit.resolved_date, 'T', ' '), '.', 1) AS timestamp),
                cast(split_part(replace(hit.closed_date, 'T', ' '), '.', 1) AS timestamp)
            )
        ) >= current_date - interval '35' day

      AND coalesce(trim(CAST(fd_bbp.type AS varchar)), trim(CAST(hit.type AS varchar)), '') <> 'Escalation - Ombudsman'

      AND coalesce(trim(CAST(hit.agent_name AS varchar)), '') <> 'No Agent'

      AND coalesce(trim(CAST(fd_bbp.insurance_category AS varchar)), trim(CAST(hit.insurance_category AS varchar)), '') <> 'Others'

      AND NOT EXISTS (
            SELECT 1
            FROM crm_l1_excluded e
            WHERE CAST(hit.ticket_id AS varchar) = e.crm_ticket_id
               OR CAST(hit.ticket_id AS varchar) = e.canonical_ticket_id
        )

    UNION ALL

    -- Branch 4: CRM base
    SELECT
        CASE lower(trim(CAST(b.last_agent_name_for_reporting AS varchar)))
            WHEN 'durga.sharma@navi.com' THEN 'Durga Prasad Sharma'
            WHEN 'afzal.sayed@navi.com' THEN 'Afzal Sayed'
            ELSE CAST(b.last_agent_name_for_reporting AS varchar)
        END AS agent_name,

        CAST(NULL AS varchar) AS Channel,

        CAST(b.ticket_state AS varchar) AS status,

        CAST(NULL AS varchar) AS policy_number,

        CAST(b.canonical_ticket_id AS varchar) AS ticket_id,

        CAST(NULL AS varchar) AS claim_id,

        CAST(b.phone_number AS varchar) AS customer_number,

        CAST(b.ticket_created_time AS timestamp)
            AS call_initiated_date_and_time,

        CAST(b.resolved_time AS timestamp)
            AS resolved_date_and_time,

        date(CAST(b.resolved_time AS timestamp)) AS resolved_date,

        CAST(b.ticket_group AS varchar) AS group_name,

        CAST(NULL AS varchar) AS "Claims-LOB",

        CAST(b.origination_reason AS varchar)
            AS call_email_disposition,

        CASE
            WHEN b.ticket_group = 'GENERAL_INSURANCE'
            THEN CAST(b.origination_sub_reason AS varchar)
            ELSE NULL
        END AS origination_sub_reason

    FROM base b

    WHERE (
            b.origination_reason IS NULL
            OR lower(trim(CAST(b.origination_reason AS varchar))) NOT IN (
                'not connected',
                'non claim related'
            )
        )

      AND (
            b.origination_sub_reason IS NULL
            OR lower(trim(CAST(b.origination_sub_reason AS varchar))) NOT IN (
                'short call',
                'others',
                'other',
                'non insurance related',
                'loan related',
                'upi related',
                'test call',
                'rnr / busy',
                'rnr/ busy',
                'rnr/switched off/ busy/ not reachalbe',
                'rnr/switched off/ busy/ not reachable',
                'rnr/switched off',
                'upi /trezo / bbps',
                'cash loan / pl',
                'home loan',
                'not connected /rnr/swithch off /busy'
            )
        )

      AND NOT EXISTS (
            SELECT 1
            FROM crm_l1_excluded e
            WHERE CAST(b.canonical_ticket_id AS varchar) = e.canonical_ticket_id
               OR CAST(b.ticket_id AS varchar) = e.crm_ticket_id
               OR CAST(b.ticket_id AS varchar) = e.canonical_ticket_id
               OR CAST(b.sub_ticket_id AS varchar) = e.crm_ticket_id
               OR CAST(b.sub_ticket_id AS varchar) = e.canonical_ticket_id
        )

      AND NOT EXISTS (
            SELECT 1
            FROM gi_outbound_calls goc
            WHERE CAST(b.canonical_ticket_id AS varchar) = CAST(goc.canonical_ticket_id AS varchar)
               OR CAST(b.ticket_id AS varchar) = CAST(goc.ticket_id AS varchar)
               OR CAST(b.sub_ticket_id AS varchar) = CAST(goc.ticket_id AS varchar)
        )

    UNION ALL

    -- Branch 5: GI Outbound Connected Calls (Query 2)
    SELECT
        CASE lower(trim(CAST(coalesce(goc.agent_email, goc.agent_name) AS varchar)))
            WHEN 'durga.sharma@navi.com' THEN 'Durga Prasad Sharma'
            WHEN 'afzal.sayed@navi.com' THEN 'Afzal Sayed'
            ELSE CAST(goc.agent_name AS varchar)
        END AS agent_name,

        CAST(goc.call_direction AS varchar) AS Channel,

        CAST(goc.ticket_state AS varchar) AS status,

        CAST(NULL AS varchar) AS policy_number,

        CAST(goc.canonical_ticket_id AS varchar) AS ticket_id,

        CAST(NULL AS varchar) AS claim_id,

        CAST(NULL AS varchar) AS customer_number,

        cast(goc.call_timestamp_ist AS timestamp) AS call_initiated_date_and_time,

        cast(goc.resolved_timestamp_ist AS timestamp) AS resolved_date_and_time,

        goc.resolved_date AS resolved_date,

        'GENERAL_INSURANCE' AS group_name,

        CAST(goc.overall_last_agent_type_for_ticket AS varchar) AS "Claims-LOB",

        CAST(goc.outbound_connection_status AS varchar) AS call_email_disposition,

        CAST(NULL AS varchar) AS origination_sub_reason

    FROM gi_outbound_calls goc
)

SELECT
    agent_name,
    Channel,
    status,
    policy_number,
    ticket_id,
    claim_id,
    customer_number,
    call_initiated_date_and_time,
    resolved_date_and_time,
    resolved_date,
    group_name,
    "Claims-LOB",
    call_email_disposition,
    origination_sub_reason

FROM (
    SELECT
        c.*,

        row_number() OVER (
            PARTITION BY ticket_id, coalesce(Channel, '')
            ORDER BY
                resolved_date_and_time DESC NULLS LAST,
                call_initiated_date_and_time DESC NULLS LAST
        ) AS rn

    FROM combined c
) ranked

WHERE rn = 1

  AND NOT (
        group_name = 'Claims Support Emails'
        AND status = 'Pending'
        AND (
            lower(trim(coalesce(call_email_disposition, ''))) LIKE '%rejection justification%'
            OR lower(trim(coalesce(call_email_disposition, ''))) LIKE '%rejection rationale%'
        )
  )

  AND (
        (
            group_name = 'Claims Support Emails'
            AND status = 'Resolved'
        )

        OR (
            call_email_disposition IS NULL
            OR lower(trim(CAST(call_email_disposition AS varchar))) NOT IN (
                'not connected',
                'non claim related'
            )
        )
    )

  AND (
        origination_sub_reason IS NULL
        OR lower(trim(CAST(origination_sub_reason AS varchar))) NOT IN (
            'short call',
            'others',
            'other',
            'non insurance related',
            'loan related',
            'upi related',
            'test call',
            'rnr / busy',
            'rnr/ busy',
            'rnr/switched off/ busy/ not reachalbe',
            'rnr/switched off/ busy/ not reachable',
            'rnr/switched off',
            'upi /trezo / bbps',
            'cash loan / pl',
            'home loan',
            'not connected /rnr/swithch off /busy'
        )
    )

  AND NOT EXISTS (
        SELECT 1
        FROM crm_l1_excluded e
        WHERE CAST(ticket_id AS varchar) = e.crm_ticket_id
           OR CAST(ticket_id AS varchar) = e.canonical_ticket_id
    )

  AND lower(trim(CAST(agent_name AS varchar))) NOT IN (
        'noor afza',
        'simon',
        'lata s',
        'deep.sahu',
        'shaziya.banu@navi.com',
        'shruthi.shetty',
        'harshitha k',
        'suraksha thimmaiah a',
        'aiyappa',
        'chandranath trivedi',
        'sridhar p',
        'prajin s',
        'ritesh kumar soni'
    )
"""

df  = get_sql_df(query)
if not df.empty:
    display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1gYNcQjkSDfUdUfwwyfcU_-wv5v4X7j7gG8-6Yg3LS9E/edit?gid=1102365521#gid=1102365521"
worksheet_name = "Master Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=2
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=2
)